# 第 33 章：表示学习、自编码器与重构误差

这个 notebook 用一个很小的光谱窗口数据集演示表示学习的最小入口：

- 先用训练集平均谱做一个最朴素的重构 baseline
- 再训练一个二维瓶颈的极小自编码器
- 比较正常样本与异常样本的重构误差
- 再把一部分正常样本当作 validation 集，校准最小 anomaly workflow
- 回到 latent 表示，理解自编码器为什么是从 PCA 走向更现代表示学习的一座桥
- 再把 dense latent 当成一个最小正常谱库，做最近邻检索与 anomaly triage
- 最后先做共享的 3-bin patch 自编码器，再推进到一个端到端的最小 Conv1d autoencoder

教学说明：这里默认不依赖 NumPy、PyTorch 或专门的深度学习库；如果环境里安装了 `torch`，最后一格会提供一个可选的极小 `nn.Linear` 自编码器对照实现。


In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

DATA_PATH = Path("../../data/small/spectral_autoencoder_demo.csv").resolve()
NUM_BINS = 12

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "split": raw["split"],
            "spectrum_role": raw["spectrum_role"],
            "pattern_label": raw["pattern_label"],
            "continuum_slope": float(raw["continuum_slope"]),
            "balmer_depth": float(raw["balmer_depth"]),
        }
        for index in range(NUM_BINS):
            parsed[f"f{index:02d}"] = float(raw[f"f{index:02d}"])
        rows.append(parsed)

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]


def spectrum_from_row(row):
    return [row[f"f{index:02d}"] for index in range(NUM_BINS)]


print(f"Loaded {len(rows)} spectra from {DATA_PATH.name}")
print("split counts:", dict(Counter(row["split"] for row in rows)))
print("role counts:", dict(Counter(row["spectrum_role"] for row in rows)))
print("training ids:", [row["sample_id"] for row in train_rows])
print("test ids:", [row["sample_id"] for row in test_rows])
print("Representative spectra:")
for sample_id in ["R01", "R08", "A01", "A02"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    rounded_flux = [round(value, 3) for value in spectrum_from_row(row)]
    print(sample_id, row["spectrum_role"], row["pattern_label"], rounded_flux)


In [ ]:
import math


def mean_vector(vectors):
    return [sum(values) / len(values) for values in zip(*vectors)]


def mean_squared_error(left, right):
    return sum((a - b) ** 2 for a, b in zip(left, right)) / len(left)



In [ ]:
def standardize(train_items, all_items):
    train_vectors = [spectrum_from_row(row) for row in train_items]
    means = mean_vector(train_vectors)
    stds = []
    for index in range(NUM_BINS):
        variance = sum((vector[index] - means[index]) ** 2 for vector in train_vectors) / len(train_vectors)
        stds.append(math.sqrt(variance) if variance > 0 else 1.0)

    scaled = []
    for row in all_items:
        vector = spectrum_from_row(row)
        scaled.append([(vector[index] - means[index]) / stds[index] for index in range(NUM_BINS)])
    return means, stds, scaled


def build_autoencoder(input_dim, latent_dim):
    encoder_weights = [
        [0.04 * (((feature_index + 1) * (latent_index + 2)) % 5 - 2) for latent_index in range(latent_dim)]
        for feature_index in range(input_dim)
    ]
    encoder_bias = [0.0 for _ in range(latent_dim)]
    decoder_weights = [
        [0.04 * (((latent_index + 1) * (feature_index + 3)) % 5 - 2) for feature_index in range(input_dim)]
        for latent_index in range(latent_dim)
    ]
    decoder_bias = [0.0 for _ in range(input_dim)]
    return encoder_weights, encoder_bias, decoder_weights, decoder_bias


def encode(vector, encoder_weights, encoder_bias):
    latent = []
    for latent_index in range(len(encoder_bias)):
        pre_activation = encoder_bias[latent_index] + sum(
            vector[feature_index] * encoder_weights[feature_index][latent_index]
            for feature_index in range(len(vector))
        )
        latent.append(math.tanh(pre_activation))
    return latent


def decode(latent, decoder_weights, decoder_bias):
    reconstructed = []
    for feature_index in range(len(decoder_bias)):
        reconstructed.append(
            decoder_bias[feature_index] + sum(
                latent[latent_index] * decoder_weights[latent_index][feature_index]
                for latent_index in range(len(latent))
            )
        )
    return reconstructed


def train_autoencoder(train_vectors, latent_dim=2, learning_rate=0.03, epochs=3500):
    input_dim = len(train_vectors[0])
    encoder_weights, encoder_bias, decoder_weights, decoder_bias = build_autoencoder(input_dim, latent_dim)
    history = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        for vector in train_vectors:
            latent = encode(vector, encoder_weights, encoder_bias)
            reconstruction = decode(latent, decoder_weights, decoder_bias)
            residual = [reconstruction[index] - vector[index] for index in range(input_dim)]
            epoch_loss += mean_squared_error(vector, reconstruction)

            gradient_output = [2.0 * value / input_dim for value in residual]
            gradient_decoder = [
                [latent[latent_index] * gradient_output[feature_index] for feature_index in range(input_dim)]
                for latent_index in range(latent_dim)
            ]
            gradient_hidden = []
            for latent_index in range(latent_dim):
                backprop_signal = sum(
                    gradient_output[feature_index] * decoder_weights[latent_index][feature_index]
                    for feature_index in range(input_dim)
                )
                gradient_hidden.append(backprop_signal * (1.0 - latent[latent_index] ** 2))
            gradient_encoder = [
                [vector[feature_index] * gradient_hidden[latent_index] for latent_index in range(latent_dim)]
                for feature_index in range(input_dim)
            ]

            for feature_index in range(input_dim):
                for latent_index in range(latent_dim):
                    encoder_weights[feature_index][latent_index] -= learning_rate * gradient_encoder[feature_index][latent_index]
            for latent_index in range(latent_dim):
                encoder_bias[latent_index] -= learning_rate * gradient_hidden[latent_index]
            for latent_index in range(latent_dim):
                for feature_index in range(input_dim):
                    decoder_weights[latent_index][feature_index] -= learning_rate * gradient_decoder[latent_index][feature_index]
            for feature_index in range(input_dim):
                decoder_bias[feature_index] -= learning_rate * gradient_output[feature_index]

        if epoch % 700 == 0:
            history.append((epoch, epoch_loss / len(train_vectors)))

    return {
        "encoder_weights": encoder_weights,
        "encoder_bias": encoder_bias,
        "decoder_weights": decoder_weights,
        "decoder_bias": decoder_bias,
        "history": history,
    }


def reconstruction_error(vector, model):
    latent = encode(vector, model["encoder_weights"], model["encoder_bias"])
    reconstruction = decode(latent, model["decoder_weights"], model["decoder_bias"])
    return mean_squared_error(vector, reconstruction), latent, reconstruction


feature_means, feature_stds, scaled_vectors = standardize(train_rows, rows)
scaled_by_id = {row["sample_id"]: vector for row, vector in zip(rows, scaled_vectors)}
train_scaled = [scaled_by_id[row["sample_id"]] for row in train_rows]
mean_reconstruction = mean_vector(train_scaled)

print("Mean-spectrum baseline on standardized features:")
for row in test_rows:
    error = mean_squared_error(scaled_by_id[row["sample_id"]], mean_reconstruction)
    print(row["sample_id"], row["spectrum_role"], round(error, 4))

normal_errors = [
    mean_squared_error(scaled_by_id[row["sample_id"]], mean_reconstruction)
    for row in test_rows
    if row["spectrum_role"] == "normal"
]
anomaly_errors = [
    mean_squared_error(scaled_by_id[row["sample_id"]], mean_reconstruction)
    for row in test_rows
    if row["spectrum_role"] == "anomaly"
]
print("mean baseline average normal error:", round(sum(normal_errors) / len(normal_errors), 4))
print("mean baseline average anomaly error:", round(sum(anomaly_errors) / len(anomaly_errors), 4))
print()
model = train_autoencoder(train_scaled)

print("Training checkpoints:")
for epoch, loss in model["history"]:
    print(f"epoch={epoch:4d} mean_train_loss={loss:.6f}")


In [ ]:
train_errors = []
for row in train_rows:
    error, latent, _ = reconstruction_error(scaled_by_id[row["sample_id"]], model)
    train_errors.append(error)

anomaly_threshold = 2.0 * max(train_errors)
print("Training reconstruction error summary:")
print("min/avg/max =", round(min(train_errors), 4), round(sum(train_errors) / len(train_errors), 4), round(max(train_errors), 4))
print("anomaly threshold =", round(anomaly_threshold, 4))
print()
print("Autoencoder reconstruction errors on test data:")
autoencoder_results = []
for row in test_rows:
    error, latent, reconstruction = reconstruction_error(scaled_by_id[row["sample_id"]], model)
    autoencoder_results.append({
        "sample_id": row["sample_id"],
        "role": row["spectrum_role"],
        "pattern_label": row["pattern_label"],
        "error": error,
        "latent": latent,
        "flagged": error > anomaly_threshold,
    })
    print(
        row["sample_id"],
        row["spectrum_role"],
        row["pattern_label"],
        "error=",
        round(error, 4),
        "flagged=",
        error > anomaly_threshold,
    )

normal_auto = [item["error"] for item in autoencoder_results if item["role"] == "normal"]
anomaly_auto = [item["error"] for item in autoencoder_results if item["role"] == "anomaly"]
print()
print("autoencoder average normal error:", round(sum(normal_auto) / len(normal_auto), 4))
print("autoencoder average anomaly error:", round(sum(anomaly_auto) / len(anomaly_auto), 4))
print("flagged anomaly ids:", [item["sample_id"] for item in autoencoder_results if item["flagged"]])


In [ ]:
print("Latent representation snapshots:")
for sample_id in ["R01", "R04", "R08", "R13", "R15"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    error, latent, reconstruction = reconstruction_error(scaled_by_id[sample_id], model)
    print(
        sample_id,
        row["pattern_label"],
        "slope=",
        row["continuum_slope"],
        "depth=",
        row["balmer_depth"],
        "latent=",
        [round(value, 3) for value in latent],
        "error=",
        round(error, 4),
    )

print("Interpretation:")
print("  The bottleneck is only two-dimensional, so the network is forced to keep the most reusable variation directions.")
print("  On this toy dataset, those directions line up roughly with continuum slope and Balmer-line depth.")
print("  That is the core representation-learning idea: learn a compact code that preserves the structure needed to reconstruct normal data.")


def pearson_correlation(x_values, y_values):
    mean_x = sum(x_values) / len(x_values)
    mean_y = sum(y_values) / len(y_values)
    numerator = sum((x - mean_x) * (y - mean_y) for x, y in zip(x_values, y_values))
    denominator_x = sum((x - mean_x) ** 2 for x in x_values)
    denominator_y = sum((y - mean_y) ** 2 for y in y_values)
    if denominator_x == 0 or denominator_y == 0:
        return 0.0
    return numerator / math.sqrt(denominator_x * denominator_y)


workflow_validation_ids = {"R05", "R06", "R12"}
workflow_train_rows = [row for row in train_rows if row["sample_id"] not in workflow_validation_ids]
workflow_validation_rows = [row for row in train_rows if row["sample_id"] in workflow_validation_ids]
workflow_rows = workflow_train_rows + workflow_validation_rows + test_rows

(
    workflow_feature_means,
    workflow_feature_stds,
    workflow_scaled_vectors,
) = standardize(workflow_train_rows, workflow_rows)
workflow_scaled_by_id = {
    row["sample_id"]: vector for row, vector in zip(workflow_rows, workflow_scaled_vectors)
}
workflow_train_scaled = [workflow_scaled_by_id[row["sample_id"]] for row in workflow_train_rows]
workflow_model = train_autoencoder(workflow_train_scaled)

workflow_validation_results = []
for row in workflow_validation_rows:
    error, latent, _ = reconstruction_error(workflow_scaled_by_id[row["sample_id"]], workflow_model)
    workflow_validation_results.append(
        {
            "sample_id": row["sample_id"],
            "pattern_label": row["pattern_label"],
            "error": error,
            "latent": latent,
        }
    )

ready_threshold = 1.5 * max(item["error"] for item in workflow_validation_results)
anomaly_candidate_threshold = 8.0 * max(item["error"] for item in workflow_validation_results)

print()
print("Validation-calibrated anomaly workflow:")
print("workflow train ids:", [row["sample_id"] for row in workflow_train_rows])
print("workflow validation ids:", [row["sample_id"] for row in workflow_validation_rows])
print(
    "validation errors:",
    [(item["sample_id"], round(item["error"], 4)) for item in workflow_validation_results],
)
print("ready threshold =", round(ready_threshold, 4))
print("anomaly-candidate threshold =", round(anomaly_candidate_threshold, 4))
print()

workflow_routes = []
workflow_route_by_id = {}
for row in test_rows:
    error, latent, _ = reconstruction_error(workflow_scaled_by_id[row["sample_id"]], workflow_model)
    if error <= ready_threshold:
        route = "normal_like"
    elif error <= anomaly_candidate_threshold:
        route = "manual_review"
    else:
        route = "anomaly_candidate"
    workflow_route_by_id[row["sample_id"]] = route
    workflow_routes.append(
        {
            "sample_id": row["sample_id"],
            "role": row["spectrum_role"],
            "pattern_label": row["pattern_label"],
            "error": error,
            "latent": latent,
            "route": route,
        }
    )
    print(
        row["sample_id"],
        row["spectrum_role"],
        row["pattern_label"],
        "error=",
        round(error, 4),
        "route=",
        route,
    )

print()
print("route counts:", dict(Counter(item["route"] for item in workflow_routes)))
print("manual review ids:", [item["sample_id"] for item in workflow_routes if item["route"] == "manual_review"])
print(
    "anomaly candidate ids:",
    [item["sample_id"] for item in workflow_routes if item["route"] == "anomaly_candidate"],
)

normal_workflow_rows = workflow_train_rows + workflow_validation_rows + [
    row for row in test_rows if row["spectrum_role"] == "normal"
]
workflow_latent_rows = []
for row in normal_workflow_rows:
    error, latent, _ = reconstruction_error(workflow_scaled_by_id[row["sample_id"]], workflow_model)
    workflow_latent_rows.append(
        {
            "sample_id": row["sample_id"],
            "continuum_slope": row["continuum_slope"],
            "balmer_depth": row["balmer_depth"],
            "latent": latent,
            "error": error,
        }
    )

latent_0 = [item["latent"][0] for item in workflow_latent_rows]
latent_1 = [item["latent"][1] for item in workflow_latent_rows]
continuum_slopes = [item["continuum_slope"] for item in workflow_latent_rows]
balmer_depths = [item["balmer_depth"] for item in workflow_latent_rows]

print()
print("latent correlation summary:")
print("  latent[0] vs continuum_slope =", round(pearson_correlation(latent_0, continuum_slopes), 3))
print("  latent[0] vs balmer_depth =", round(pearson_correlation(latent_0, balmer_depths), 3))
print("  latent[1] vs continuum_slope =", round(pearson_correlation(latent_1, continuum_slopes), 3))
print("  latent[1] vs balmer_depth =", round(pearson_correlation(latent_1, balmer_depths), 3))
print("Workflow reading:")
print("  R16 lands in manual_review because it extends the red, shallow-line edge of the normal manifold.")
print("  A04 also lands in manual_review because a broad absorption trough is unusual, but not as extreme as the spike and burst anomalies.")


def extract_patches(vector, patch_width=3):
    return [vector[start:start + patch_width] for start in range(len(vector) - patch_width + 1)]


def reconstruct_from_patches(patch_reconstructions, output_length, patch_width=3):
    sums = [0.0 for _ in range(output_length)]
    counts = [0 for _ in range(output_length)]
    for start, patch in enumerate(patch_reconstructions):
        for offset, value in enumerate(patch):
            sums[start + offset] += value
            counts[start + offset] += 1
    return [sums[index] / counts[index] for index in range(output_length)]


def train_patch_autoencoder(train_vectors, patch_width=3, latent_dim=2, learning_rate=0.04, epochs=2600):
    train_patches = []
    for vector in train_vectors:
        train_patches.extend(extract_patches(vector, patch_width))
    model = train_autoencoder(train_patches, latent_dim=latent_dim, learning_rate=learning_rate, epochs=epochs)
    model["patch_width"] = patch_width
    model["patch_count"] = len(train_patches)
    return model


def patch_reconstruction_summary(vector, model):
    patch_errors = []
    patch_latents = []
    patch_reconstructions = []
    for patch in extract_patches(vector, model["patch_width"]):
        error, latent, reconstruction = reconstruction_error(patch, model)
        patch_errors.append(error)
        patch_latents.append(latent)
        patch_reconstructions.append(reconstruction)
    full_reconstruction = reconstruct_from_patches(
        patch_reconstructions,
        output_length=len(vector),
        patch_width=model["patch_width"],
    )
    peak_patch_error = max(patch_errors)
    peak_patch_start = patch_errors.index(peak_patch_error)
    return {
        "spectrum_error": mean_squared_error(vector, full_reconstruction),
        "peak_patch_error": peak_patch_error,
        "peak_patch_start": peak_patch_start,
        "patch_errors": patch_errors,
        "patch_latents": patch_latents,
        "reconstruction": full_reconstruction,
    }


patch_model = train_patch_autoencoder(workflow_train_scaled)

print()
print("Sliding-patch autoencoder as a conv-style extension:")
print("  The same 3-bin encoder/decoder is reused at every position.")
print("  This approximates the local-receptive-field + weight-sharing intuition behind a 1D convolutional autoencoder.")
print("patch training checkpoints:")
for epoch, loss in patch_model["history"]:
    print(f"epoch={epoch:4d} mean_patch_loss={loss:.6f}")
print("patch count seen during training =", patch_model["patch_count"])

patch_validation_results = []
for row in workflow_validation_rows:
    summary = patch_reconstruction_summary(workflow_scaled_by_id[row["sample_id"]], patch_model)
    patch_validation_results.append(
        {
            "sample_id": row["sample_id"],
            "spectrum_error": summary["spectrum_error"],
            "peak_patch_error": summary["peak_patch_error"],
        }
    )

conv_ready_threshold = 1.5 * max(item["spectrum_error"] for item in patch_validation_results)
conv_peak_patch_ready_threshold = 1.5 * max(item["peak_patch_error"] for item in patch_validation_results)
conv_peak_patch_anomaly_threshold = 3.5 * max(item["peak_patch_error"] for item in patch_validation_results)

print()
print("Validation-calibrated local-patch workflow:")
print(
    "validation summaries:",
    [
        (
            item["sample_id"],
            round(item["spectrum_error"], 4),
            round(item["peak_patch_error"], 4),
        )
        for item in patch_validation_results
    ],
)
print("spectrum ready threshold =", round(conv_ready_threshold, 4))
print("peak-patch ready threshold =", round(conv_peak_patch_ready_threshold, 4))
print("peak-patch anomaly threshold =", round(conv_peak_patch_anomaly_threshold, 4))

patch_routes = []
for row in test_rows:
    summary = patch_reconstruction_summary(workflow_scaled_by_id[row["sample_id"]], patch_model)
    if summary["spectrum_error"] <= conv_ready_threshold and summary["peak_patch_error"] <= conv_peak_patch_ready_threshold:
        route = "normal_like"
    elif summary["peak_patch_error"] <= conv_peak_patch_anomaly_threshold:
        route = "manual_review"
    else:
        route = "anomaly_candidate"
    peak_patch_start = summary["peak_patch_start"]
    peak_window = f"f{peak_patch_start:02d}-f{peak_patch_start + patch_model['patch_width'] - 1:02d}"
    patch_routes.append(
        {
            "sample_id": row["sample_id"],
            "dense_route": workflow_route_by_id[row["sample_id"]],
            "route": route,
            "spectrum_error": summary["spectrum_error"],
            "peak_patch_error": summary["peak_patch_error"],
            "peak_window": peak_window,
        }
    )
    print(
        row["sample_id"],
        row["pattern_label"],
        "spectrum_error=",
        round(summary["spectrum_error"], 4),
        "peak_patch_error=",
        round(summary["peak_patch_error"], 4),
        "peak_window=",
        peak_window,
        "route=",
        route,
    )

print()
print("route counts:", dict(Counter(item["route"] for item in patch_routes)))
print("manual review ids:", [item["sample_id"] for item in patch_routes if item["route"] == "manual_review"])
print("anomaly candidate ids:", [item["sample_id"] for item in patch_routes if item["route"] == "anomaly_candidate"])
print("Route changes relative to the dense bottleneck workflow:")
for item in patch_routes:
    if item["dense_route"] != item["route"]:
        print(" ", item["sample_id"], item["dense_route"], "->", item["route"], "at", item["peak_window"])
print("Conv-style reading:")
print("  R16 stays in manual_review because its red-edge shape sits just outside the normal local patch range.")
print("  A04 moves from manual_review to anomaly_candidate because the wide trough keeps triggering high local mismatch across patches f07-f09.")


def build_conv1d_autoencoder(kernel_width=4, filters=3, stride=1):
    encoder_weights = [
        [0.05 * (((filter_index + 2) * (offset + 1)) % 5 - 2) for offset in range(kernel_width)]
        for filter_index in range(filters)
    ]
    encoder_bias = [0.0 for _ in range(filters)]
    decoder_weights = [
        [0.05 * (((filter_index + 3) * (offset + 2)) % 5 - 2) for offset in range(kernel_width)]
        for filter_index in range(filters)
    ]
    decoder_bias = [0.0 for _ in range(NUM_BINS)]
    positions = list(range(0, NUM_BINS - kernel_width + 1, stride))
    return {
        "kernel_width": kernel_width,
        "filters": filters,
        "stride": stride,
        "positions": positions,
        "encoder_weights": encoder_weights,
        "encoder_bias": encoder_bias,
        "decoder_weights": decoder_weights,
        "decoder_bias": decoder_bias,
    }


def conv1d_encode(vector, model):
    latents = []
    for filter_index in range(model["filters"]):
        channel = []
        for start in model["positions"]:
            pre_activation = model["encoder_bias"][filter_index] + sum(
                model["encoder_weights"][filter_index][offset] * vector[start + offset]
                for offset in range(model["kernel_width"])
            )
            channel.append(math.tanh(pre_activation))
        latents.append(channel)
    return latents


def conv1d_decode(latents, model):
    reconstruction = list(model["decoder_bias"])
    for filter_index in range(model["filters"]):
        for position_index, start in enumerate(model["positions"]):
            latent_value = latents[filter_index][position_index]
            for offset in range(model["kernel_width"]):
                reconstruction[start + offset] += model["decoder_weights"][filter_index][offset] * latent_value
    return reconstruction


def conv1d_reconstruction_error(vector, model):
    latents = conv1d_encode(vector, model)
    reconstruction = conv1d_decode(latents, model)
    return mean_squared_error(vector, reconstruction), latents, reconstruction


def train_conv1d_autoencoder(train_vectors, kernel_width=4, filters=3, stride=1, learning_rate=0.02, epochs=4200):
    model = build_conv1d_autoencoder(kernel_width=kernel_width, filters=filters, stride=stride)
    history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for vector in train_vectors:
            latents = conv1d_encode(vector, model)
            reconstruction = conv1d_decode(latents, model)
            output_gradient = [
                2.0 * (reconstruction[index] - vector[index]) / len(vector)
                for index in range(len(vector))
            ]
            epoch_loss += mean_squared_error(vector, reconstruction)

            decoder_weight_gradient = [
                [0.0 for _ in range(model["kernel_width"])]
                for _ in range(model["filters"])
            ]
            latent_gradient = [
                [0.0 for _ in model["positions"]]
                for _ in range(model["filters"])
            ]
            for filter_index in range(model["filters"]):
                for position_index, start in enumerate(model["positions"]):
                    latent_value = latents[filter_index][position_index]
                    backprop_signal = 0.0
                    for offset in range(model["kernel_width"]):
                        decoder_weight_gradient[filter_index][offset] += output_gradient[start + offset] * latent_value
                        backprop_signal += output_gradient[start + offset] * model["decoder_weights"][filter_index][offset]
                    latent_gradient[filter_index][position_index] = backprop_signal * (1.0 - latent_value ** 2)

            encoder_weight_gradient = [
                [0.0 for _ in range(model["kernel_width"])]
                for _ in range(model["filters"])
            ]
            encoder_bias_gradient = [0.0 for _ in range(model["filters"])]
            for filter_index in range(model["filters"]):
                for position_index, start in enumerate(model["positions"]):
                    grad = latent_gradient[filter_index][position_index]
                    encoder_bias_gradient[filter_index] += grad
                    for offset in range(model["kernel_width"]):
                        encoder_weight_gradient[filter_index][offset] += grad * vector[start + offset]

            for filter_index in range(model["filters"]):
                for offset in range(model["kernel_width"]):
                    model["encoder_weights"][filter_index][offset] -= learning_rate * encoder_weight_gradient[filter_index][offset]
                    model["decoder_weights"][filter_index][offset] -= learning_rate * decoder_weight_gradient[filter_index][offset]
                model["encoder_bias"][filter_index] -= learning_rate * encoder_bias_gradient[filter_index]
            for index in range(len(vector)):
                model["decoder_bias"][index] -= learning_rate * output_gradient[index]

        if epoch % 840 == 0:
            history.append((epoch, epoch_loss / len(train_vectors)))

    model["history"] = history
    return model


def dominant_residual_window(vector, reconstruction, window_width=4):
    window_errors = []
    for start in range(len(vector) - window_width + 1):
        window_errors.append(
            sum((vector[start + offset] - reconstruction[start + offset]) ** 2 for offset in range(window_width))
            / window_width
        )
    peak_error = max(window_errors)
    peak_start = window_errors.index(peak_error)
    return peak_error, f"f{peak_start:02d}-f{peak_start + window_width - 1:02d}"


conv1d_model = train_conv1d_autoencoder(workflow_train_scaled)

print()
print("End-to-end Conv1d autoencoder extension:")
print("  Shared filters now see the full spectrum during training, not isolated patches.")
print("  This is the notebook's smallest end-to-end Conv1d autoencoder.")
print("training checkpoints:")
for epoch, loss in conv1d_model["history"]:
    print(f"epoch={epoch:4d} mean_train_loss={loss:.6f}")
print(
    "learned encoder kernels:",
    [[round(value, 3) for value in kernel] for kernel in conv1d_model["encoder_weights"]],
)

conv1d_validation_results = []
for row in workflow_validation_rows:
    error, latents, reconstruction = conv1d_reconstruction_error(workflow_scaled_by_id[row["sample_id"]], conv1d_model)
    peak_window_error, peak_window = dominant_residual_window(
        workflow_scaled_by_id[row["sample_id"]],
        reconstruction,
        window_width=conv1d_model["kernel_width"],
    )
    conv1d_validation_results.append(
        {
            "sample_id": row["sample_id"],
            "error": error,
            "peak_window_error": peak_window_error,
            "peak_window": peak_window,
        }
    )

conv1d_ready_threshold = 1.3 * max(item["error"] for item in conv1d_validation_results)
conv1d_anomaly_threshold = 2.5 * max(item["error"] for item in conv1d_validation_results)

print()
print("Validation-calibrated end-to-end Conv1d workflow:")
print(
    "validation summaries:",
    [
        (
            item["sample_id"],
            round(item["error"], 4),
            round(item["peak_window_error"], 4),
            item["peak_window"],
        )
        for item in conv1d_validation_results
    ],
)
print("ready threshold =", round(conv1d_ready_threshold, 4))
print("anomaly-candidate threshold =", round(conv1d_anomaly_threshold, 4))

conv1d_routes = []
for row in test_rows:
    error, latents, reconstruction = conv1d_reconstruction_error(workflow_scaled_by_id[row["sample_id"]], conv1d_model)
    peak_window_error, peak_window = dominant_residual_window(
        workflow_scaled_by_id[row["sample_id"]],
        reconstruction,
        window_width=conv1d_model["kernel_width"],
    )
    if error <= conv1d_ready_threshold:
        route = "normal_like"
    elif error <= conv1d_anomaly_threshold:
        route = "manual_review"
    else:
        route = "anomaly_candidate"
    conv1d_routes.append(
        {
            "sample_id": row["sample_id"],
            "dense_route": workflow_route_by_id[row["sample_id"]],
            "route": route,
            "error": error,
            "peak_window_error": peak_window_error,
            "peak_window": peak_window,
        }
    )
    print(
        row["sample_id"],
        row["pattern_label"],
        "spectrum_error=",
        round(error, 4),
        "peak_window_error=",
        round(peak_window_error, 4),
        "peak_window=",
        peak_window,
        "route=",
        route,
    )

conv1d_normal_errors = [item["error"] for item in conv1d_routes[:4]]
conv1d_anomaly_errors = [item["error"] for item in conv1d_routes[4:]]
print()
print("average normal error =", round(sum(conv1d_normal_errors) / len(conv1d_normal_errors), 4))
print("average anomaly error =", round(sum(conv1d_anomaly_errors) / len(conv1d_anomaly_errors), 4))
print("route counts:", dict(Counter(item["route"] for item in conv1d_routes)))
print("manual review ids:", [item["sample_id"] for item in conv1d_routes if item["route"] == "manual_review"])
print("anomaly candidate ids:", [item["sample_id"] for item in conv1d_routes if item["route"] == "anomaly_candidate"])
print("Route changes relative to the dense bottleneck workflow:")
for item in conv1d_routes:
    if item["dense_route"] != item["route"]:
        print(" ", item["sample_id"], item["dense_route"], "->", item["route"], "at", item["peak_window"])
print("Conv1d reading:")
print("  R16 remains in manual_review because the model reconstructs most bins well, but still sees the shallow red-edge variant as slightly outside validation support.")
print("  A04 moves to anomaly_candidate once the shared filters must reconstruct the whole trough end-to-end and the full-spectrum error crosses the anomaly threshold.")


def latent_distance(left, right):
    return math.sqrt(sum((left_value - right_value) ** 2 for left_value, right_value in zip(left, right)))


workflow_dense_latent_by_id = {}
for row in workflow_train_rows + workflow_validation_rows + test_rows:
    error, latent, _ = reconstruction_error(workflow_scaled_by_id[row["sample_id"]], workflow_model)
    workflow_dense_latent_by_id[row["sample_id"]] = latent


def retrieve_normal_neighbors(row, reference_rows, top_k=3):
    candidate_latent = workflow_dense_latent_by_id[row["sample_id"]]
    ranked = []
    for reference_row in reference_rows:
        ranked.append(
            {
                "sample_id": reference_row["sample_id"],
                "pattern_label": reference_row["pattern_label"],
                "continuum_slope": reference_row["continuum_slope"],
                "balmer_depth": reference_row["balmer_depth"],
                "distance": latent_distance(candidate_latent, workflow_dense_latent_by_id[reference_row["sample_id"]]),
            }
        )
    ranked.sort(key=lambda item: item["distance"])
    return ranked[:top_k]


workflow_retrieval_validation = []
for row in workflow_validation_rows:
    neighbors = retrieve_normal_neighbors(row, workflow_train_rows)
    nearest = neighbors[0]
    workflow_retrieval_validation.append(
        {
            "sample_id": row["sample_id"],
            "true_pattern": row["pattern_label"],
            "retrieved_pattern": nearest["pattern_label"],
            "nearest_distance": nearest["distance"],
        }
    )

latent_support_threshold = 1.1 * max(item["nearest_distance"] for item in workflow_retrieval_validation)
validation_family_accuracy = sum(
    item["true_pattern"] == item["retrieved_pattern"] for item in workflow_retrieval_validation
) / len(workflow_retrieval_validation)

normal_test_rows = [row for row in test_rows if row["spectrum_role"] == "normal"]
normal_test_retrieval = []
for row in normal_test_rows:
    neighbors = retrieve_normal_neighbors(row, workflow_train_rows)
    normal_test_retrieval.append(
        {
            "sample_id": row["sample_id"],
            "true_pattern": row["pattern_label"],
            "retrieved_pattern": neighbors[0]["pattern_label"],
            "nearest_distance": neighbors[0]["distance"],
        }
    )
normal_test_family_accuracy = sum(
    item["true_pattern"] == item["retrieved_pattern"] for item in normal_test_retrieval
) / len(normal_test_retrieval)

workflow_conv1d_route_by_id = {item["sample_id"]: item for item in conv1d_routes}


def retrieval_triage_label(conv1d_route, within_support):
    if conv1d_route == "normal_like":
        return "retrieval_ready"
    if conv1d_route == "manual_review" and within_support:
        return "normal_manifold_edge"
    if conv1d_route == "manual_review":
        return "manifold_shift_review"
    if within_support:
        return "localized_anomaly_over_known_family"
    return "manifold_shift_anomaly_candidate"


workflow_retrieval_triage = []
for row in test_rows:
    neighbors = retrieve_normal_neighbors(row, workflow_train_rows)
    nearest = neighbors[0]
    conv1d_summary = workflow_conv1d_route_by_id[row["sample_id"]]
    within_support = nearest["distance"] <= latent_support_threshold
    retrieval_route = retrieval_triage_label(conv1d_summary["route"], within_support)
    workflow_retrieval_triage.append(
        {
            "sample_id": row["sample_id"],
            "spectrum_role": row["spectrum_role"],
            "pattern_label": row["pattern_label"],
            "retrieved_pattern": nearest["pattern_label"],
            "nearest_distance": nearest["distance"],
            "within_support": within_support,
            "conv1d_route": conv1d_summary["route"],
            "conv1d_peak_window": conv1d_summary["peak_window"],
            "triage": retrieval_route,
            "neighbors": neighbors,
        }
    )

anomaly_retrieval_rows = [item for item in workflow_retrieval_triage if item["spectrum_role"] == "anomaly"]

print()
print("Latent retrieval and anomaly triage extension:")
print("  The dense bottleneck now acts as a tiny reference library of normal spectra.")
print("  Retrieval answers which normal family a candidate most resembles; residual workflow still decides whether that resemblance is trustworthy.")
print(
    "validation nearest-neighbor summaries:",
    [
        (
            item["sample_id"],
            item["retrieved_pattern"],
            round(item["nearest_distance"], 4),
        )
        for item in workflow_retrieval_validation
    ],
)
print("latent support threshold =", round(latent_support_threshold, 4))
print("validation family accuracy =", round(validation_family_accuracy, 3))
print("normal-test family accuracy =", round(normal_test_family_accuracy, 3))
print(
    "anomalies still inside latent support =",
    f"{sum(item['within_support'] for item in anomaly_retrieval_rows)}/{len(anomaly_retrieval_rows)}",
)
print()
print("Combined retrieval + conv1d triage:")
for item in workflow_retrieval_triage:
    print(
        item["sample_id"],
        item["spectrum_role"],
        item["pattern_label"],
        "retrieved=",
        item["retrieved_pattern"],
        "distance=",
        round(item["nearest_distance"], 4),
        "within_support=",
        item["within_support"],
        "conv1d_peak=",
        item["conv1d_peak_window"],
        "triage=",
        item["triage"],
    )
print("triage counts:", dict(Counter(item["triage"] for item in workflow_retrieval_triage)))
print("retrieval-ready ids:", [item["sample_id"] for item in workflow_retrieval_triage if item["triage"] == "retrieval_ready"])
print("normal-manifold-edge ids:", [item["sample_id"] for item in workflow_retrieval_triage if item["triage"] == "normal_manifold_edge"])
print(
    "localized anomaly ids:",
    [item["sample_id"] for item in workflow_retrieval_triage if item["triage"] == "localized_anomaly_over_known_family"],
)
print(
    "manifold-shift anomaly ids:",
    [item["sample_id"] for item in workflow_retrieval_triage if item["triage"] == "manifold_shift_anomaly_candidate"],
)

print()
print("Representative retrieval lookups:")
for sample_id in ["R16", "A02", "A04"]:
    row = next(item for item in test_rows if item["sample_id"] == sample_id)
    summary = next(item for item in workflow_retrieval_triage if item["sample_id"] == sample_id)
    print()
    print(sample_id, row["pattern_label"], summary["triage"])
    print(
        "top neighbors =",
        [
            (
                item["sample_id"],
                item["pattern_label"],
                round(item["distance"], 4),
            )
            for item in summary["neighbors"]
        ],
    )
    print(
        "retrieved family =",
        summary["retrieved_pattern"],
        "within_support =",
        summary["within_support"],
        "conv1d peak window =",
        summary["conv1d_peak_window"],
    )

print("Interpretation:")
print("  All four normal test spectra retrieve the expected normal family, so the 2D latent space is already useful as a small representation library.")
print("  Three of the four anomalies still sit inside that latent support, which shows why retrieval alone is not a sufficient anomaly detector when the bottleneck mostly tracks global continuum and line-depth structure.")
print("  R16 stays close to the red_continuum_shallow_line family and therefore looks like a normal-manifold edge case, while A02, A03, and A04 retrieve familiar global families but still break them through localized residual structure.")
print("  A01 is the strongest manifold-shift case: it not only triggers the anomaly route, but also lands outside the validation-calibrated latent support radius.")


In [ ]:
try:
    import matplotlib.pyplot as plt

    sample_ids = ["R16", "A04"]
    figure, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    for axis, sample_id in zip(axes, sample_ids):
        row = next(item for item in rows if item["sample_id"] == sample_id)
        scaled = workflow_scaled_by_id[sample_id]
        _, _, scaled_reconstruction = reconstruction_error(scaled, workflow_model)
        original = spectrum_from_row(row)
        reconstructed = [
            scaled_reconstruction[index] * workflow_feature_stds[index] + workflow_feature_means[index]
            for index in range(NUM_BINS)
        ]
        axis.plot(original, marker="o", label="original")
        axis.plot(reconstructed, marker="s", linestyle="--", label="reconstruction")
        axis.set_title(f"{sample_id} ({workflow_route_by_id[sample_id]})")
        axis.set_xlabel("flux bin")
        axis.grid(alpha=0.3)
    axes[0].set_ylabel("normalized flux")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")


In [ ]:
try:
    import torch
    from torch import nn

    torch.manual_seed(0)

    train_tensor = torch.tensor(train_scaled, dtype=torch.float32)

    class TinyAutoencoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(nn.Linear(NUM_BINS, 2), nn.Tanh())
            self.decoder = nn.Linear(2, NUM_BINS)

        def forward(self, inputs):
            latent = self.encoder(inputs)
            return self.decoder(latent)

    net = TinyAutoencoder()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.02)
    criterion = nn.MSELoss()

    for _ in range(300):
        optimizer.zero_grad()
        reconstruction = net(train_tensor)
        loss = criterion(reconstruction, train_tensor)
        loss.backward()
        optimizer.step()

    print("Optional PyTorch reconstruction loss:", round(float(loss.detach()), 6))
except Exception as exc:
    print(f"Optional PyTorch comparison skipped: {exc}")


In [ ]:
from part5_delivery_exports import export_ch33_autoencoder_delivery

export_ch33_autoencoder_delivery(globals())
